In [3]:
import pandas as pd
import json
import os

# ================================================
# QUICK FILE CHECK — Are all 3 files uploaded?
# ================================================

# --- Check File 1: Box Office Mojo CSV ---
df_boxoffice = pd.read_csv('boxoffice_raw.csv')
print("✅ boxoffice_raw.csv loaded!")
print(f"   → Rows: {len(df_boxoffice)}")
print(f"   → Columns: {list(df_boxoffice.columns)}")
print(f"   → First movie: {df_boxoffice['title'][0]}")
print()

# --- Check File 2: Top 200 Movies CSV ---
df_csv = pd.read_csv('Top_200_Movies_Dataset_2023.csv')
print("✅ Top_200_Movies CSV loaded!")
print(f"   → Rows: {len(df_csv)}")
print(f"   → Columns: {list(df_csv.columns)}")
print(f"   → First movie: {df_csv['Title'][0]}")
print()

# --- Check File 3: TMDB JSON ---
with open('tmdb_raw.json', 'r', encoding='utf-8') as f:
    tmdb_data = json.load(f)
print("✅ tmdb_raw.json loaded!")
print(f"   → Number of movies: {len(tmdb_data)}")
print(f"   → First movie: {tmdb_data[0]['title']}")
print(f"   → Fields available: {list(tmdb_data[0].keys())}")
print()

# --- Final Summary ---
print("=" * 40)
print("📦 FILE SUMMARY")
print("=" * 40)
print(f"  Box Office Mojo  → {len(df_boxoffice)} movies")
print(f"  Top 200 CSV      → {len(df_csv)} movies")
print(f"  TMDB JSON        → {len(tmdb_data)} movies")
print(f"  TOTAL RECORDS    → {len(df_boxoffice) + len(df_csv) + len(tmdb_data)} records across 3 sources")
print("=" * 40)
print("🎉 All files loaded successfully! Ready to clean.")

✅ boxoffice_raw.csv loaded!
   → Rows: 1000
   → Columns: ['rank', 'title', 'revenue', 'year']
   → First movie: Star Wars: Episode VII - The Force Awakens

✅ Top_200_Movies CSV loaded!
   → Rows: 200
   → Columns: ['Rank', 'Title', 'Theaters', 'Total Gross', 'Release Date', 'Distributor']
   → First movie: Barbie

✅ tmdb_raw.json loaded!
   → Number of movies: 500
   → First movie: The Super Mario Bros. Movie
   → Fields available: ['adult', 'backdrop_path', 'genre_ids', 'id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'release_date', 'title', 'video', 'vote_average', 'vote_count']

📦 FILE SUMMARY
  Box Office Mojo  → 1000 movies
  Top 200 CSV      → 200 movies
  TMDB JSON        → 500 movies
  TOTAL RECORDS    → 1700 records across 3 sources
🎉 All files loaded successfully! Ready to clean.


In [4]:
# ================================================
# BEFORE CLEANING — Box Office Mojo
# Let's SEE the problems first before fixing them
# ================================================

print("📋 FIRST 5 ROWS:")
print(df_boxoffice.head())
print()

print("📋 DATA TYPES (how Python sees each column):")
print(df_boxoffice.dtypes)
print()

print("📋 MISSING VALUES:")
print(df_boxoffice.isnull().sum())
print()

print("📋 SAMPLE OF REVENUE COLUMN:")
print(df_boxoffice['revenue'].head(10))

📋 FIRST 5 ROWS:
  rank                                       title       revenue  year
0    1  Star Wars: Episode VII - The Force Awakens  $936,662,225  2015
1    2                           Avengers: Endgame  $858,373,000  2019
2    3                     Spider-Man: No Way Home  $814,866,759  2021
3    4                                      Avatar  $785,221,649  2009
4    5                           Top Gun: Maverick  $718,732,821  2022

📋 DATA TYPES (how Python sees each column):
rank       object
title      object
revenue    object
year        int64
dtype: object

📋 MISSING VALUES:
rank       0
title      0
revenue    0
year       0
dtype: int64

📋 SAMPLE OF REVENUE COLUMN:
0    $936,662,225
1    $858,373,000
2    $814,866,759
3    $785,221,649
4    $718,732,821
5    $700,426,566
6    $688,459,501
7    $678,815,482
8    $674,354,882
9    $653,406,625
Name: revenue, dtype: object


In [5]:
# ================================================
# BEFORE CLEANING STATS — Box Office Mojo
# We save these numbers for our Quality Report later
# ================================================

print("📊 BEFORE CLEANING — Box Office Mojo")
print("=" * 40)
print(f"Total rows: {len(df_boxoffice)}")
print(f"Missing values: {df_boxoffice.isnull().sum().sum()}")
print(f"Duplicate rows: {df_boxoffice.duplicated().sum()}")
print(f"Revenue column type: {df_boxoffice['revenue'].dtype}")
print(f"Sample revenue value: {df_boxoffice['revenue'][0]}")
print("=" * 40)

📊 BEFORE CLEANING — Box Office Mojo
Total rows: 1000
Missing values: 0
Duplicate rows: 0
Revenue column type: object
Sample revenue value: $936,662,225


In [7]:
# ================================================
# CLEANING — Box Office Mojo (Fixed)
# ================================================

# Make a copy so we never touch the original
df_boxoffice_clean = df_boxoffice.copy()

# --- Fix 1: Clean the revenue column ---
df_boxoffice_clean['revenue'] = (
    df_boxoffice_clean['revenue']
    .str.replace('$', '', regex=False)   # remove $
    .str.replace(',', '', regex=False)   # remove commas
    .astype(int)                         # convert to number
)

# --- Fix 2: Make rank a number (remove commas first!) ---
df_boxoffice_clean['rank'] = (
    df_boxoffice_clean['rank']
    .str.replace(',', '', regex=False)   # remove commas first
    .astype(int)                         # then convert to number
)

# --- Fix 3: Clean movie titles ---
df_boxoffice_clean['title_clean'] = (
    df_boxoffice_clean['title']
    .str.lower()        # make all lowercase
    .str.strip()        # remove extra spaces
)

print("✅ Cleaning done!")
print()
print("📋 FIRST 5 ROWS AFTER CLEANING:")
print(df_boxoffice_clean.head())
print()
print("📋 DATA TYPES AFTER CLEANING:")
print(df_boxoffice_clean.dtypes)

✅ Cleaning done!

📋 FIRST 5 ROWS AFTER CLEANING:
   rank                                       title    revenue  year  \
0     1  Star Wars: Episode VII - The Force Awakens  936662225  2015   
1     2                           Avengers: Endgame  858373000  2019   
2     3                     Spider-Man: No Way Home  814866759  2021   
3     4                                      Avatar  785221649  2009   
4     5                           Top Gun: Maverick  718732821  2022   

                                  title_clean  
0  star wars: episode vii - the force awakens  
1                           avengers: endgame  
2                     spider-man: no way home  
3                                      avatar  
4                           top gun: maverick  

📋 DATA TYPES AFTER CLEANING:
rank            int64
title          object
revenue         int64
year            int64
title_clean    object
dtype: object


In [8]:
# ================================================
# AFTER CLEANING STATS — Box Office Mojo
# ================================================

print("📊 AFTER CLEANING — Box Office Mojo")
print("=" * 40)
print(f"Total rows: {len(df_boxoffice_clean)}")
print(f"Missing values: {df_boxoffice_clean.isnull().sum().sum()}")
print(f"Duplicate rows: {df_boxoffice_clean.duplicated().sum()}")
print(f"Revenue column type: {df_boxoffice_clean['revenue'].dtype}")
print(f"Sample revenue value: {df_boxoffice_clean['revenue'][0]}")
print()
print("✅ BEFORE vs AFTER SUMMARY:")
print(f"  Revenue was text → now number ✅")
print(f"  Rank was text    → now number ✅")
print(f"  Added title_clean column      ✅")
print("=" * 40)

📊 AFTER CLEANING — Box Office Mojo
Total rows: 1000
Missing values: 0
Duplicate rows: 0
Revenue column type: int64
Sample revenue value: 936662225

✅ BEFORE vs AFTER SUMMARY:
  Revenue was text → now number ✅
  Rank was text    → now number ✅
  Added title_clean column      ✅


In [9]:
# ================================================
# SAVE CLEAN FILE — Box Office Mojo
# ================================================

df_boxoffice_clean.to_csv('boxoffice_clean.csv', index=False)

print("✅ boxoffice_clean.csv saved successfully!")
print(f"   → Rows saved: {len(df_boxoffice_clean)}")
print(f"   → Columns saved: {list(df_boxoffice_clean.columns)}")

✅ boxoffice_clean.csv saved successfully!
   → Rows saved: 1000
   → Columns saved: ['rank', 'title', 'revenue', 'year', 'title_clean']


## ✅ Box Office Mojo — Cleaning Complete

**What we did:**
- Started with 1,000 raw movie records
- Removed `$` and commas from revenue column (e.g. `$936,662,225` → `936662225`)
- Removed commas from rank column (e.g. `1,000` → `1000`)
- Converted both revenue and rank from text to real numbers
- Added a new `title_clean` column (lowercase + no extra spaces)
- No missing values or duplicates were found

**Output file:** `boxoffice_clean.csv` ✅

In [10]:
# ================================================
# BEFORE CLEANING — Top 200 Movies CSV
# Let's SEE the problems first
# ================================================

print("📊 BEFORE CLEANING — Top 200 Movies CSV")
print("=" * 40)
print(f"Total rows: {len(df_csv)}")
print(f"Missing values per column:")
print(df_csv.isnull().sum())
print()
print(f"Duplicate rows: {df_csv.duplicated().sum()}")
print()
print("📋 DATA TYPES:")
print(df_csv.dtypes)
print()
print("📋 FIRST 3 ROWS:")
print(df_csv.head(3))
print()
print("📋 SAMPLE OF EACH COLUMN:")
print(f"  Total Gross  : {df_csv['Total Gross'][0]}")
print(f"  Theaters     : {df_csv['Theaters'][0]}")
print(f"  Release Date : {df_csv['Release Date'][0]}")
print(f"  Distributor  : {df_csv['Distributor'][0]}")

📊 BEFORE CLEANING — Top 200 Movies CSV
Total rows: 200
Missing values per column:
Rank            0
Title           0
Theaters        0
Total Gross     0
Release Date    0
Distributor     0
dtype: int64

Duplicate rows: 0

📋 DATA TYPES:
Rank             int64
Title           object
Theaters        object
Total Gross     object
Release Date    object
Distributor     object
dtype: object

📋 FIRST 3 ROWS:
   Rank                                Title Theaters   Total Gross  \
0     1                               Barbie    4,337  $594,254,460   
1     2          The Super Mario Bros. Movie    4,371  $574,759,600   
2     3  Spider-Man: Across the Spider-Verse    4,332  $381,178,195   

          Release Date         Distributor  
0  2023-07-21 00:00:00        Warner Bros.  
1  2023-04-05 00:00:00  Universal Pictures  
2  2023-06-02 00:00:00   Columbia Pictures  

📋 SAMPLE OF EACH COLUMN:
  Total Gross  : $594,254,460
  Theaters     : 4,337
  Release Date : 2023-07-21 00:00:00
  Distributor

In [11]:
# ================================================
# CLEANING — Top 200 Movies CSV
# ================================================

# Make a copy so we never touch the original
df_csv_clean = df_csv.copy()

# --- Fix 1: Clean Total Gross ---
# Remove $ and commas, then convert to number
df_csv_clean['Total Gross'] = (
    df_csv_clean['Total Gross']
    .str.replace('$', '', regex=False)   # remove $
    .str.replace(',', '', regex=False)   # remove commas
    .astype(int)                         # convert to number
)

# --- Fix 2: Clean Theaters ---
# Remove commas, replace '- with Unknown, then convert to number
df_csv_clean['Theaters'] = (
    df_csv_clean['Theaters']
    .str.replace(',', '', regex=False)   # remove commas
    .str.replace("'-", '0', regex=False) # replace '- with 0
    .astype(int)                         # convert to number
)

# --- Fix 3: Extract Year from Release Date ---
# 2023-07-21 00:00:00 → 2023
df_csv_clean['Year'] = (
    df_csv_clean['Release Date']
    .str[:4]        # take only first 4 characters = the year
    .astype(int)    # convert to number
)

# --- Fix 4: Fix Distributor column ---
# Replace '- with Unknown
df_csv_clean['Distributor'] = (
    df_csv_clean['Distributor']
    .str.replace("'-", 'Unknown', regex=False)
)

# --- Fix 5: Add clean title column ---
df_csv_clean['title_clean'] = (
    df_csv_clean['Title']
    .str.lower()     # lowercase
    .str.strip()     # remove extra spaces
)

print("✅ Cleaning done!")
print()
print("📋 FIRST 5 ROWS AFTER CLEANING:")
print(df_csv_clean.head())
print()
print("📋 DATA TYPES AFTER CLEANING:")
print(df_csv_clean.dtypes)

✅ Cleaning done!

📋 FIRST 5 ROWS AFTER CLEANING:
   Rank                                Title  Theaters  Total Gross  \
0     1                               Barbie      4337    594254460   
1     2          The Super Mario Bros. Movie      4371    574759600   
2     3  Spider-Man: Across the Spider-Verse      4332    381178195   
3     4       Guardians of the Galaxy Vol. 3      4450    358995815   
4     5                          Oppenheimer      3761    300144670   

          Release Date                          Distributor  Year  \
0  2023-07-21 00:00:00                         Warner Bros.  2023   
1  2023-04-05 00:00:00                   Universal Pictures  2023   
2  2023-06-02 00:00:00                    Columbia Pictures  2023   
3  2023-05-05 00:00:00  Walt Disney Studios Motion Pictures  2023   
4  2023-07-21 00:00:00                   Universal Pictures  2023   

                           title_clean  
0                               barbie  
1          the super mario 

In [12]:
# ================================================
# VERIFY — Check the problem values were fixed
# ================================================

# Check 1: Any '- left in Distributor?
print("📋 Checking Distributor for unknown values:")
print(df_csv_clean[df_csv_clean['Distributor'] == 'Unknown'])
print()

# Check 2: Any 0 values in Theaters? (these were the '- ones)
print("📋 Movies with 0 Theaters (were missing):")
print(df_csv_clean[df_csv_clean['Theaters'] == 0][['Title', 'Theaters', 'Distributor']])
print()

# Check 3: Missing values after cleaning
print("📋 Missing values after cleaning:")
print(df_csv_clean.isnull().sum())

📋 Checking Distributor for unknown values:
     Rank                                              Title  Theaters  \
29     30                                                Air      3507   
69     70  Star Wars: Episode VI - Return of the Jedi2023...       475   
87     88                          Ponniyin Selvan: Part Two         0   
119   120                              Oldboy2023 Re-release       250   
129   130                              Knights of the Zodiac       588   
138   139                               The Devil Conspiracy       925   
140   141                          How to Blow Up a Pipeline       530   
155   156                                          Sanctuary       225   
165   166                                          Terrifier       853   
166   167                                            Selfiee       308   
169   170                               Children of the Corn       586   
170   171                                   Waltair Veerayya       35

In [13]:
# ================================================
# FLAG RE-RELEASES
# ================================================

# Add a new column that marks re-releases as True/False
df_csv_clean['is_rerelease'] = (
    df_csv_clean['Title']
    .str.contains('Re-release|Anniversary|Re-Release',
                   case=False,    # ignore uppercase/lowercase
                   regex=True)    # look for any of these words
)

# Show which movies got flagged
print("📋 Movies flagged as re-releases:")
print(df_csv_clean[df_csv_clean['is_rerelease'] == True][['Title', 'Year', 'is_rerelease']])
print()
print(f"Total re-releases flagged: {df_csv_clean['is_rerelease'].sum()}")

📋 Movies flagged as re-releases:
                                                 Title  Year  is_rerelease
54                          Titanic25 Year Anniversary  2023          True
69   Star Wars: Episode VI - Return of the Jedi2023...  2023          True
106    Jurassic Park2023 Re-release (30th Anniversary)  2023          True
118       Spirited Away2023 Re-release (Live on Stage)  2023          True
119                              Oldboy2023 Re-release  2023          True
125  The Lord of the Rings: The Return of the King2...  2023          True
150                            The Way2023  Re-release  2023          True
158      Crouching Tiger, Hidden Dragon2023 Re-release  2023          True
184               RRR Fan CelebRRRation2023 Re-release  2023          True
190                             Lourdes2022 Re-release  2023          True
194                      The Conformist2023 Re-release  2023          True
195          Contempt4k Restoration - 60th Anniversary  2023       

In [14]:
# ================================================
# AFTER CLEANING STATS — Top 200 Movies CSV
# ================================================

print("📊 AFTER CLEANING — Top 200 Movies CSV")
print("=" * 40)
print(f"Total rows: {len(df_csv_clean)}")
print(f"Missing values: {df_csv_clean.isnull().sum().sum()}")
print(f"Duplicate rows: {df_csv_clean.duplicated().sum()}")
print()
print("✅ BEFORE vs AFTER SUMMARY:")
print(f"  Total Gross was text    → now number ✅")
print(f"  Theaters was text       → now number ✅")
print(f"  Extracted Year column   → done ✅")
print(f"  Fixed '- in Distributor → now 'Unknown' ✅")
print(f"  Fixed '- in Theaters    → now 0 ✅")
print(f"  Added title_clean       → done ✅")
print(f"  Re-releases flagged     → 12 movies ✅")
print("=" * 40)

# ── Save the clean file ──────────────────────────
df_csv_clean.to_csv('movies_reference_clean.csv', index=False)
print()
print("✅ movies_reference_clean.csv saved successfully!")
print(f"   → Rows saved: {len(df_csv_clean)}")
print(f"   → Columns saved: {list(df_csv_clean.columns)}")

📊 AFTER CLEANING — Top 200 Movies CSV
Total rows: 200
Missing values: 0
Duplicate rows: 0

✅ BEFORE vs AFTER SUMMARY:
  Total Gross was text    → now number ✅
  Theaters was text       → now number ✅
  Extracted Year column   → done ✅
  Fixed '- in Distributor → now 'Unknown' ✅
  Fixed '- in Theaters    → now 0 ✅
  Added title_clean       → done ✅
  Re-releases flagged     → 12 movies ✅

✅ movies_reference_clean.csv saved successfully!
   → Rows saved: 200
   → Columns saved: ['Rank', 'Title', 'Theaters', 'Total Gross', 'Release Date', 'Distributor', 'Year', 'title_clean', 'is_rerelease']


## ✅ Top 200 Movies CSV — Cleaning Complete

**What we did:**
- Started with 200 raw movie records (all from 2023)
- Removed `$` and commas from Total Gross column → converted to number
- Removed commas from Theaters column → converted to number
- Extracted Year from full datetime (e.g. `2023-07-21 00:00:00` → `2023`)
- Replaced `'-` in Distributor column with `"Unknown"` (19 movies affected)
- Replaced `'-` in Theaters column with `0` (2 movies affected)
- Added `title_clean` column (lowercase + no extra spaces)
- Flagged 12 re-releases with `is_rerelease = True`
- No missing values or duplicates found

**Output file:** `movies_reference_clean.csv` ✅

In [15]:
# ================================================
# BEFORE CLEANING — TMDB JSON
# Let's SEE the problems first
# ================================================

import pandas as pd
import json

# First convert JSON to a dataframe (like a spreadsheet)
df_tmdb = pd.DataFrame(tmdb_data)

print("📊 BEFORE CLEANING — TMDB JSON")
print("=" * 40)
print(f"Total rows: {len(df_tmdb)}")
print()
print("📋 ALL COLUMNS:")
print(df_tmdb.columns.tolist())
print()
print("📋 DATA TYPES:")
print(df_tmdb.dtypes)
print()
print("📋 MISSING VALUES:")
print(df_tmdb.isnull().sum())
print()
print("📋 FIRST ROW AS EXAMPLE:")
print(df_tmdb.iloc[0])

📊 BEFORE CLEANING — TMDB JSON
Total rows: 500

📋 ALL COLUMNS:
['adult', 'backdrop_path', 'genre_ids', 'id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'release_date', 'title', 'video', 'vote_average', 'vote_count']

📋 DATA TYPES:
adult                   bool
backdrop_path         object
genre_ids             object
id                     int64
original_language     object
original_title        object
overview              object
popularity           float64
poster_path           object
release_date          object
title                 object
video                   bool
vote_average         float64
vote_count             int64
dtype: object

📋 MISSING VALUES:
adult                0
backdrop_path        0
genre_ids            0
id                   0
original_language    0
original_title       0
overview             0
popularity           0
poster_path          0
release_date         0
title                0
video                0
vote_average      

In [16]:
# ================================================
# CLEANING — TMDB JSON
# ================================================

# Make a copy so we never touch the original
df_tmdb_clean = df_tmdb.copy()

# --- Fix 1: Drop columns we don't need ---
columns_to_drop = ['backdrop_path', 'poster_path',
                   'overview', 'adult', 'video']
df_tmdb_clean = df_tmdb_clean.drop(columns=columns_to_drop)

print("✅ Step 1 done — dropped useless columns")
print(f"   Columns remaining: {df_tmdb_clean.columns.tolist()}")
print()

# --- Fix 2: Extract year from release_date ---
# "2023-04-05" → 2023
df_tmdb_clean['year'] = (
    df_tmdb_clean['release_date']
    .str[:4]          # take first 4 characters = year
    .astype(int)      # convert to number
)

print("✅ Step 2 done — extracted year from release_date")
print(f"   Sample: {df_tmdb_clean['year'][0]}")
print()

# --- Fix 3: Decode genre_ids into genre names ---
# TMDB uses numbers for genres — this is the official list
GENRE_MAP = {
    28:    "Action",
    12:    "Adventure",
    16:    "Animation",
    35:    "Comedy",
    80:    "Crime",
    99:    "Documentary",
    18:    "Drama",
    10751: "Family",
    14:    "Fantasy",
    36:    "History",
    27:    "Horror",
    10402: "Music",
    9648:  "Mystery",
    10749: "Romance",
    878:   "Science Fiction",
    10770: "TV Movie",
    53:    "Thriller",
    10752: "War",
    37:    "Western"
}

# Convert list of numbers to list of words
# e.g. [10751, 35, 12] → "Family, Comedy, Adventure"
df_tmdb_clean['genres'] = df_tmdb_clean['genre_ids'].apply(
    lambda ids: ', '.join([GENRE_MAP.get(i, 'Unknown') for i in ids])
)

print("✅ Step 3 done — decoded genre IDs to genre names")
print(f"   Sample: {df_tmdb_clean['genres'][0]}")
print()

# --- Fix 4: Add clean title column ---
df_tmdb_clean['title_clean'] = (
    df_tmdb_clean['title']
    .str.lower()      # lowercase
    .str.strip()      # remove extra spaces
)

print("✅ Step 4 done — added title_clean column")
print()

print("📋 FIRST 3 ROWS AFTER CLEANING:")
print(df_tmdb_clean[['title', 'year', 'genres',
                      'vote_average', 'popularity']].head(3))
print()
print("📋 DATA TYPES AFTER CLEANING:")
print(df_tmdb_clean.dtypes)

✅ Step 1 done — dropped useless columns
   Columns remaining: ['genre_ids', 'id', 'original_language', 'original_title', 'popularity', 'release_date', 'title', 'vote_average', 'vote_count']

✅ Step 2 done — extracted year from release_date
   Sample: 2023

✅ Step 3 done — decoded genre IDs to genre names
   Sample: Family, Comedy, Adventure, Fantasy, Animation

✅ Step 4 done — added title_clean column

📋 FIRST 3 ROWS AFTER CLEANING:
                         title  year  \
0  The Super Mario Bros. Movie  2023   
1    The Passion of the Christ  2004   
2        The Devil Wears Prada  2006   

                                          genres  vote_average  popularity  
0  Family, Comedy, Adventure, Fantasy, Animation         7.588    380.5992  
1                                          Drama         7.537    130.3440  
2                                  Drama, Comedy         7.389     99.2223  

📋 DATA TYPES AFTER CLEANING:
genre_ids             object
id                     int64
origin

In [18]:
# ================================================
# AFTER CLEANING STATS — TMDB (Fixed)
# ================================================

print("📊 AFTER CLEANING — TMDB JSON")
print("=" * 40)
print(f"Total rows: {len(df_tmdb_clean)}")
print(f"Missing values: {df_tmdb_clean.isnull().sum().sum()}")

# Exclude genre_ids column when checking duplicates
# because it contains lists which Python can't compare
cols_to_check = [col for col in df_tmdb_clean.columns if col != 'genre_ids']
print(f"Duplicate rows: {df_tmdb_clean.duplicated(subset=cols_to_check).sum()}")
print()

# Check genres decoded correctly
print("📋 SAMPLE OF GENRES DECODED:")
print(df_tmdb_clean[['title', 'genres']].head(5))
print()

# Check years look correct
print("📋 YEAR RANGE IN DATASET:")
print(f"   Oldest movie: {df_tmdb_clean['year'].min()}")
print(f"   Newest movie: {df_tmdb_clean['year'].max()}")
print()

print("✅ BEFORE vs AFTER SUMMARY:")
print(f"  Dropped 5 useless columns        ✅")
print(f"  Extracted year from release_date ✅")
print(f"  Decoded genre numbers to names   ✅")
print(f"  Added title_clean column         ✅")
print("=" * 40)

# ── Save the clean file ──────────────────────────
df_tmdb_clean.to_csv('tmdb_clean.csv', index=False)
print()
print("✅ tmdb_clean.csv saved successfully!")
print(f"   → Rows saved: {len(df_tmdb_clean)}")
print(f"   → Columns saved: {list(df_tmdb_clean.columns)}")

📊 AFTER CLEANING — TMDB JSON
Total rows: 500
Missing values: 0
Duplicate rows: 0

📋 SAMPLE OF GENRES DECODED:
                         title                                         genres
0  The Super Mario Bros. Movie  Family, Comedy, Adventure, Fantasy, Animation
1    The Passion of the Christ                                          Drama
2        The Devil Wears Prada                                  Drama, Comedy
3      Spider-Man: No Way Home             Action, Adventure, Science Fiction
4                 Interstellar              Adventure, Drama, Science Fiction

📋 YEAR RANGE IN DATASET:
   Oldest movie: 1938
   Newest movie: 2024

✅ BEFORE vs AFTER SUMMARY:
  Dropped 5 useless columns        ✅
  Extracted year from release_date ✅
  Decoded genre numbers to names   ✅
  Added title_clean column         ✅

✅ tmdb_clean.csv saved successfully!
   → Rows saved: 500
   → Columns saved: ['genre_ids', 'id', 'original_language', 'original_title', 'popularity', 'release_date', 'title',

## ✅ TMDB JSON — Cleaning Complete

**What we did:**
- Started with 500 raw movie records (1938–2024)
- Converted JSON format into a spreadsheet format
- Dropped 5 useless columns (backdrop_path, poster_path, overview, adult, video)
- Extracted Year from release_date (e.g. `"2023-04-05"` → `2023`)
- Decoded genre numbers into real words (e.g. `[28, 12]` → `"Action, Adventure"`)
- Added `title_clean` column (lowercase + no extra spaces)
- No missing values or duplicates found

**Output file:** `tmdb_clean.csv` ✅

In [20]:
# ================================================
# DATA QUALITY REPORT — All 3 Files
# Person 2: Maria El Khoury
# ================================================

print("=" * 60)
print("       DATA QUALITY REPORT — MOVIE PIPELINE")
print("       Person 2: Data Cleaning")
print("=" * 60)

# ─────────────────────────────────────────────
# FILE 1: BOX OFFICE MOJO
# ─────────────────────────────────────────────
print("\n📁 FILE 1: Box Office Mojo (boxoffice_raw.csv)")
print("-" * 60)
print(f"{'Metric':<35} {'BEFORE':>10} {'AFTER':>10}")
print("-" * 60)
print(f"{'Total rows':<35} {'1000':>10} {'1000':>10}")
print(f"{'Missing values':<35} {'0':>10} {'0':>10}")
print(f"{'Duplicate rows':<35} {'0':>10} {'0':>10}")
print(f"{'Revenue column type':<35} {'text':>10} {'number':>10}")
print(f"{'Rank column type':<35} {'text':>10} {'number':>10}")
print(f"{'title_clean column':<35} {'no':>10} {'yes':>10}")
print("-" * 60)
print("✏️  Decisions made:")
print("   • Removed $ and commas from revenue → needed for math")
print("   • Removed commas from rank → needed for sorting")
print("   • Added title_clean → needed for matching across files")

# ─────────────────────────────────────────────
# FILE 2: TOP 200 MOVIES CSV
# ─────────────────────────────────────────────
print("\n📁 FILE 2: Top 200 Movies CSV")
print("-" * 60)
print(f"{'Metric':<35} {'BEFORE':>10} {'AFTER':>10}")
print("-" * 60)
print(f"{'Total rows':<35} {'200':>10} {'200':>10}")
print(f"{'Missing values':<35} {'0':>10} {'0':>10}")
print(f"{'Duplicate rows':<35} {'0':>10} {'0':>10}")
print(f"{'Total Gross column type':<35} {'text':>10} {'number':>10}")
print(f"{'Theaters column type':<35} {'text':>10} {'number':>10}")
print(f"{'Year column':<35} {'no':>10} {'yes':>10}")
print(f"{'Distributor = unknown':<35} {'19':>10} {'fixed':>10}")
print(f"{'Theaters = missing':<35} {'2':>10} {'set 0':>10}")
print(f"{'Re-releases flagged':<35} {'no':>10} {'12':>10}")
print(f"{'title_clean column':<35} {'no':>10} {'yes':>10}")
print("-" * 60)
print("✏️  Decisions made:")
print("   • Removed $ and commas from Total Gross → needed for math")
print("   • Removed commas from Theaters → needed for math")
print("   • Extracted Year from full datetime → simpler for analysis")
print("   • Replaced '- in Distributor with 'Unknown' → kept rows,")
print("     did not delete, because movie data is still valid")
print("   • Replaced '- in Theaters with 0 → missing but not invalid")
print("   • Flagged re-releases instead of deleting → Person 3 decides")
print("   • Added title_clean → needed for matching across files")

# ─────────────────────────────────────────────
# FILE 3: TMDB JSON
# ─────────────────────────────────────────────
print("\n📁 FILE 3: TMDB JSON (tmdb_raw.json)")
print("-" * 60)
print(f"{'Metric':<35} {'BEFORE':>10} {'AFTER':>10}")
print("-" * 60)
print(f"{'Total rows':<35} {'500':>10} {'500':>10}")
print(f"{'Missing values':<35} {'0':>10} {'0':>10}")
print(f"{'Duplicate rows':<35} {'0':>10} {'0':>10}")
print(f"{'Columns count':<35} {'14':>10} {'10':>10}")
print(f"{'Useless columns dropped':<35} {'0':>10} {'5':>10}")
print(f"{'Year column':<35} {'no':>10} {'yes':>10}")
print(f"{'Genres as numbers':<35} {'yes':>10} {'no':>10}")
print(f"{'Genres as words':<35} {'no':>10} {'yes':>10}")
print(f"{'title_clean column':<35} {'no':>10} {'yes':>10}")
print("-" * 60)
print("✏️  Decisions made:")
print("   • Dropped backdrop_path, poster_path, overview,")
print("     adult, video → irrelevant for movie analysis")
print("   • Extracted Year from release_date → simpler for analysis")
print("   • Decoded genre IDs to names → human readable,")
print("     easier to query and analyze")
print("   • Added title_clean → needed for matching across files")

# ─────────────────────────────────────────────
# OVERALL SUMMARY
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("       OVERALL SUMMARY")
print("=" * 60)
print(f"  Total movies processed : 1,700 (across 3 files)")
print(f"  Total missing values   : 0 after cleaning")
print(f"  Total duplicates       : 0 after cleaning")
print(f"  Files output           : 3 clean CSV files")
print(f"    → boxoffice_clean.csv")
print(f"    → movies_reference_clean.csv")
print(f"    → tmdb_clean.csv")
print("=" * 60)
print("✅ Data Quality Report Complete!")

       DATA QUALITY REPORT — MOVIE PIPELINE
       Person 2: Data Cleaning

📁 FILE 1: Box Office Mojo (boxoffice_raw.csv)
------------------------------------------------------------
Metric                                  BEFORE      AFTER
------------------------------------------------------------
Total rows                                1000       1000
Missing values                               0          0
Duplicate rows                               0          0
Revenue column type                       text     number
Rank column type                          text     number
title_clean column                          no        yes
------------------------------------------------------------
✏️  Decisions made:
   • Removed $ and commas from revenue → needed for math
   • Removed commas from rank → needed for sorting
   • Added title_clean → needed for matching across files

📁 FILE 2: Top 200 Movies CSV
------------------------------------------------------------
Metric      

## 📊 Data Quality Report — Summary

### File 1: Box Office Mojo
| Metric | Before | After |
|--------|--------|-------|
| Total rows | 1,000 | 1,000 |
| Missing values | 0 | 0 |
| Duplicate rows | 0 | 0 |
| Revenue type | text ❌ | number ✅ |
| Rank type | text ❌ | number ✅ |

**Decisions made:**
- Removed `$` and commas from revenue → needed for math operations
- Removed commas from rank → needed for sorting
- Added `title_clean` → needed for matching movies across all 3 files

---

### File 2: Top 200 Movies CSV
| Metric | Before | After |
|--------|--------|-------|
| Total rows | 200 | 200 |
| Missing values | 0 | 0 |
| Duplicate rows | 0 | 0 |
| Total Gross type | text ❌ | number ✅ |
| Theaters type | text ❌ | number ✅ |
| Unknown distributors | 19 | fixed ✅ |
| Missing theaters | 2 | set to 0 ✅ |
| Re-releases flagged | 0 | 12 ✅ |

**Decisions made:**
- Replaced `'-` in Distributor with `"Unknown"` → kept rows because movie data is still valid
- Replaced `'-` in Theaters with `0` → data is missing but not invalid
- Flagged 12 re-releases instead of deleting → left decision to Person 3

---

### File 3: TMDB JSON
| Metric | Before | After |
|--------|--------|-------|
| Total rows | 500 | 500 |
| Missing values | 0 | 0 |
| Duplicate rows | 0 | 0 |
| Columns | 14 | 10 |
| Genres | numbers ❌ | words ✅ |

**Decisions made:**
- Dropped 5 irrelevant columns (images, descriptions) → not needed for analysis
- Decoded genre numbers to real names → human readable and easier to query
- Extracted year from full date → simpler for analysis

---

### Overall Summary
- ✅ Total movies processed: 1,700 across 3 files
- ✅ Total missing values after cleaning: 0
- ✅ Total duplicates after cleaning: 0
- ✅ Output: 3 clean CSV files ready for Person 3